In [1]:
%pip install "medprep-starter[medical] @ git+https://github.com/Z-bros/Med_Im_Prepper.git"

  Cloning https://github.com/Z-bros/Med_Im_Prepper.git to /tmp/pip-install-h1a0brhj/medprep-starter_123d43c3e3194be1a7e1258fb68627aa
  Running command git clone --filter=blob:none --quiet https://github.com/Z-bros/Med_Im_Prepper.git /tmp/pip-install-h1a0brhj/medprep-starter_123d43c3e3194be1a7e1258fb68627aa
  Resolved https://github.com/Z-bros/Med_Im_Prepper.git to commit 9941c68c3ad771c751c5be9e4c30e7fff82ef9b4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for medprep-starter: filename=medprep_starter-0.1.0-py3-none-any.whl size=12117 sha256=c0e42caf659f14cbed51be972623b40df54a76328896fcbe2a17131c62fa153a
  Stored in directory: /tmp/pip-ephem-wheel-cache-e9e001ym/wheels/de/f5/8a/e994093450acec394d6073104c38f80d0b53cbec90fc216d93
Successfully built medprep-starter
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import medprep

ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

print("medprep version:", medprep.__version__)
print("Dataset available:", ROOT.exists())

medprep version: 0.1.0
Dataset available: True


# Load the Tables

In [3]:
train = pd.read_csv(ROOT / "train.csv")
series = pd.read_csv(ROOT / "train_series.csv")
submission = pd.read_csv(ROOT / "sample_submission.csv")

label_columns = [
    column for column in submission.columns
    if column != "StudyInstanceUID"
]

print("Training studies:", train["StudyInstanceUID"].nunique())
print("Training series:", series["SeriesInstanceUID"].nunique())
print("Target conditions:", len(label_columns))

Training studies: 4407
Training series: 24371
Target conditions: 12


# Checking the Labels

In [4]:
known_labels = train[label_columns].notna()

summary = pd.Series({
    "Fully labelled studies": known_labels.all(axis=1).sum(),
    "Partially labelled studies": (
        known_labels.any(axis=1) & ~known_labels.all(axis=1)
    ).sum(),
    "Studies without labels": (~known_labels.any(axis=1)).sum(),
    "Studies with nonempty reports": (
        train["Report"].fillna("").str.strip().ne("")
    ).sum(),
})

display(summary.to_frame("Count"))

,Count
Fully labelled studies,58
Partially labelled studies,0
Studies without labels,4349
Studies with nonempty reports,4407


# Exploring Series Data

In [5]:
series_counts = (
    series.groupby("StudyInstanceUID")
    .size()
    .reindex(train["StudyInstanceUID"], fill_value=0)
)

display(series_counts.describe().to_frame("Series per study"))
display(series["Anatomical_Plane"].value_counts(dropna=False))

,Series per study
count,4407.000000
mean,5.530066
std,1.393826
min,3.000000
25%,5.000000
50%,5.000000
75%,6.000000
max,14.000000


Anatomical_Plane
Sagittal    9864
Coronal     8609
Axial       5898
Name: count, dtype: int64

Confirms and Check whether training studies have at least one sagittal, coronal, and axial series in the metadata.

In [6]:
plane_counts = pd.crosstab(
    series["StudyInstanceUID"],
    series["Anatomical_Plane"],
).reindex(
    index=train["StudyInstanceUID"],
    columns=["Sagittal", "Coronal", "Axial"],
    fill_value=0,
)

plane_present = plane_counts.gt(0)

display(pd.DataFrame({
    "Studies with plane": plane_present.sum(),
    "Studies missing plane": (~plane_present).sum(),
}))

print(
    "Studies with all three planes:",
    plane_present.all(axis=1).sum(),
)

display(plane_counts.head())

,Studies with plane,Studies missing plane
Anatomical_Plane,,
Sagittal,4407,0
Coronal,4407,0
Axial,4407,0


Studies with all three planes: 4407


Anatomical_Plane,Sagittal,Coronal,Axial
StudyInstanceUID,,,
1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260,2,2,1
1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585,2,2,1
1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231,2,2,1
1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764,3,4,3
1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542,2,1,2


# Further checks for each view has at least 1 matching series

In [7]:
sequence_summary = (
    series.groupby(
        ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"],
        dropna=False,
    )
    .size()
    .rename("Series count")
    .reset_index()
    .sort_values(
        ["Anatomical_Plane", "Series count"],
        ascending=[True, False],
    )
)

display(sequence_summary)

,Anatomical_Plane,Fluid_Sensitive,Fat_Suppression,Series count
1,Axial,1,1,4719
0,Axial,0,0,1179
3,Coronal,1,1,4624
2,Coronal,0,0,3985
4,Sagittal,0,0,5197
5,Sagittal,1,1,4667


In [8]:
selected = series[
    series["Fluid_Sensitive"].eq(1)
    & series["Fat_Suppression"].eq(1)
]

coverage = pd.crosstab(
    selected["StudyInstanceUID"],
    selected["Anatomical_Plane"],
).reindex(
    index=train["StudyInstanceUID"],
    columns=["Sagittal", "Coronal", "Axial"],
    fill_value=0,
)

display(pd.DataFrame({
    "Studies with matching series": coverage.gt(0).sum(),
    "Studies without matching series": coverage.eq(0).sum(),
}))

print(
    "Studies with matching series in all three planes:",
    coverage.gt(0).all(axis=1).sum(),
)

,Studies with matching series,Studies without matching series
Anatomical_Plane,,
Sagittal,4150,257
Coronal,4248,159
Axial,4407,0


Studies with matching series in all three planes: 3991


A useful distinction that: every study has all three planes, but some lack a series with both flags set to 1 in sagittal or coronal views.

For later modelling, a selection rule that requires this series type in every plane would exclude 416 studies. 

Then --> should retain those studies and design a fallback or explicitly handle the unavailable series type.

# The Take
- 4,407 studies, 24,371 series, and 12 target conditions.
- 58 fully labelled studies; 4,349 without official labels; reports available for all.
- Every study has sagittal, coronal, and axial series according to the CSV.
- Both series flags match throughout the supplied metadata.
- 3,991 studies have a both-flags-1 series in every plane; 416 need an alternative selection strategy.
- These findings describe metadata coverage; image readability and quality remain unchecked.